In [2]:
# =========================================================
# RPDA8412 RESEARCH PROJECT
# NOTEBOOK 06: FEATURE ENGINEERING
# =========================================================

from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

FINAL_DIR = PROJECT_ROOT / "data" / "final"
TABLES_DIR = PROJECT_ROOT / "tables"

TABLES_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(
    FINAL_DIR / "daily_volatility_engineered_provisional.csv",
    parse_dates=["Date"]
)

df = df.sort_values("Date").reset_index(drop=True)

print("=" * 60)
print("FEATURE ENGINEERING")
print("=" * 60)

print("Original shape:", df.shape)
print("Date range:", df["Date"].min(), "to", df["Date"].max())

display(df.head())

FEATURE ENGINEERING
Original shape: (4024, 15)
Date range: 2010-01-04 00:00:00 to 2025-12-31 00:00:00


,Date,Adj_Close,Close,High,Low,Open,Volume,GPR_lag1,EPU_lag1,Log_Return,Squared_Return,RV_Historical_20D,RV_Future_1D,RV_Future_5D,RV_Future_20D
0,2010-01-04,1132.989990,1132.989990,1133.869995,1116.560059,1116.560059,3991400000,91.471298,216.95,NaN,NaN,NaN,0.003111,0.006103,0.046068
1,2010-01-05,1136.520020,1136.520020,1136.630005,1129.660034,1132.660034,2491020000,177.117065,150.35,0.003111,9.677276e-06,NaN,0.000545,0.010789,0.046290
2,2010-01-06,1137.140015,1137.140015,1139.189941,1133.949951,1135.709961,4972660000,122.525620,151.16,0.000545,2.974305e-07,NaN,0.003993,0.013596,0.056065
3,2010-01-07,1141.689941,1141.689941,1142.459961,1131.319946,1136.270020,5270680000,161.588669,95.95,0.003993,1.594579e-05,NaN,0.002878,0.013221,0.055997
4,2010-01-08,1144.979980,1144.979980,1145.390015,1136.219971,1140.520020,4389590000,102.262291,92.20,0.002878,8.280484e-06,NaN,0.001745,0.016880,0.056627


In [3]:
# =========================================================
# STEP 1: MARKET-BASED FEATURES
# =========================================================

# Previous trading day's log return
df["Return_Lag1"] = df["Log_Return"].shift(1)

# Magnitude of current daily return
df["Absolute_Return"] = df["Log_Return"].abs()

# Historical realised volatility
for window in [5, 10, 20, 60]:

    df[f"RV_Historical_{window}D"] = np.sqrt(
        df["Squared_Return"].rolling(
            window=window,
            min_periods=window
        ).sum()
    )

# Previous trading day's 20-day volatility
df["RV_20D_Lag1"] = (
    df["RV_Historical_20D"].shift(1)
)

print("=" * 60)
print("MARKET FEATURES CREATED")
print("=" * 60)

market_features = [
    "Return_Lag1",
    "Absolute_Return",
    "RV_Historical_5D",
    "RV_Historical_10D",
    "RV_Historical_20D",
    "RV_Historical_60D",
    "RV_20D_Lag1"
]

display(df[market_features].describe().T)

print("\nMissing values:")
print(df[market_features].isna().sum())

MARKET FEATURES CREATED


,count,mean,std,min,25%,50%,75%,max
Return_Lag1,4022.0,0.000449,0.010943,-0.127652,-0.003808,0.000702,0.005676,0.090895
Absolute_Return,4023.0,0.007225,0.008229,0.000000,0.001973,0.004837,0.009742,0.127652
RV_Historical_5D,4019.0,0.019598,0.014697,0.001362,0.010600,0.016385,0.024053,0.200982
RV_Historical_10D,4014.0,0.028612,0.019562,0.004404,0.017069,0.023857,0.034562,0.231988
RV_Historical_20D,4004.0,0.041530,0.026071,0.009526,0.026326,0.035147,0.049002,0.270908
RV_Historical_60D,3964.0,0.075035,0.040045,0.024806,0.053053,0.062702,0.084571,0.310117
RV_20D_Lag1,4003.0,0.041534,0.026073,0.009526,0.026333,0.035147,0.049020,0.270908



Missing values:
Return_Lag1           2
Absolute_Return       1
RV_Historical_5D      5
RV_Historical_10D    10
RV_Historical_20D    20
RV_Historical_60D    60
RV_20D_Lag1          21
dtype: int64


In [4]:
# =========================================================
# STEP 2: GEOPOLITICAL RISK FEATURES
# =========================================================

# Existing provisional GPR level
df["GPR_Level"] = df["GPR_lag1"]

# Previous trading-day GPR level
df["GPR_Previous"] = df["GPR_Level"].shift(1)

# Change in GPR
df["GPR_Change"] = (
    df["GPR_Level"] - df["GPR_Previous"]
)

# Percentage change
# Replace zero denominators with NaN
df["GPR_Pct_Change"] = (
    df["GPR_Level"] /
    df["GPR_Previous"].replace(0, np.nan)
    - 1
)

# Five-observation rolling average
df["GPR_MA5"] = (
    df["GPR_Level"].rolling(5).mean()
)

# Twenty-observation rolling average
df["GPR_MA20"] = (
    df["GPR_Level"].rolling(20).mean()
)

gpr_features = [
    "GPR_Level",
    "GPR_Change",
    "GPR_Pct_Change",
    "GPR_MA5",
    "GPR_MA20"
]

print("=" * 60)
print("GPR FEATURES CREATED")
print("=" * 60)

display(df[gpr_features].describe().T)

print("\nMissing values:")
print(df[gpr_features].isna().sum())

GPR FEATURES CREATED


,count,mean,std,min,25%,50%,75%,max
GPR_Level,4024.0,106.215976,51.519556,0.000000,71.635078,97.428459,130.606281,540.827393
GPR_Change,4023.0,0.031673,51.450717,-239.197372,-31.232483,0.534805,30.894995,300.494354
GPR_Pct_Change,4022.0,0.177171,0.894042,-1.000000,-0.271436,0.004909,0.380762,16.659091
GPR_MA5,4020.0,106.188571,38.187362,37.214302,80.597544,97.679031,123.465932,431.177051
GPR_MA20,4005.0,106.149466,32.397895,54.611151,83.587574,98.230521,120.349275,339.208426



Missing values:
GPR_Level          0
GPR_Change         1
GPR_Pct_Change     2
GPR_MA5            4
GPR_MA20          19
dtype: int64


In [5]:
# =========================================================
# STEP 3: ECONOMIC POLICY UNCERTAINTY FEATURES
# =========================================================

# Existing provisional EPU level
df["EPU_Level"] = df["EPU_lag1"]

# Previous trading-day EPU level
df["EPU_Previous"] = df["EPU_Level"].shift(1)

# Absolute change
df["EPU_Change"] = (
    df["EPU_Level"] - df["EPU_Previous"]
)

# Percentage change
df["EPU_Pct_Change"] = (
    df["EPU_Level"] /
    df["EPU_Previous"].replace(0, np.nan)
    - 1
)

# Rolling averages
df["EPU_MA5"] = (
    df["EPU_Level"].rolling(5).mean()
)

df["EPU_MA20"] = (
    df["EPU_Level"].rolling(20).mean()
)

epu_features = [
    "EPU_Level",
    "EPU_Change",
    "EPU_Pct_Change",
    "EPU_MA5",
    "EPU_MA20"
]

print("=" * 60)
print("EPU FEATURES CREATED")
print("=" * 60)

display(df[epu_features].describe().T)

print("\nMissing values:")
print(df[epu_features].isna().sum())

EPU FEATURES CREATED


,count,mean,std,min,25%,50%,75%,max
EPU_Level,4024.0,147.749292,115.585078,4.050000,76.570000,113.010000,174.700000,1026.380000
EPU_Change,4023.0,0.022086,85.133384,-952.780000,-36.885000,-1.210000,36.895000,872.820000
EPU_Pct_Change,4023.0,0.180474,1.173070,-0.983941,-0.282735,-0.012913,0.376136,52.005566
EPU_MA5,4020.0,147.628413,100.633532,30.282000,86.428500,116.730000,164.279500,670.284000
EPU_MA20,4005.0,147.277979,95.074036,43.955000,90.142000,122.227000,161.035000,576.553500



Missing values:
EPU_Level          0
EPU_Change         1
EPU_Pct_Change     1
EPU_MA5            4
EPU_MA20          19
dtype: int64


In [6]:
# =========================================================
# STEP 4: DEFINE FEATURE CONFIGURATIONS
# =========================================================

MARKET_FEATURES = [
    "Return_Lag1",
    "Absolute_Return",
    "RV_Historical_5D",
    "RV_Historical_10D",
    "RV_Historical_20D",
    "RV_Historical_60D",
    "RV_20D_Lag1"
]

GPR_FEATURES = [
    "GPR_Level",
    "GPR_Change",
    "GPR_MA5",
    "GPR_MA20"
]

EPU_FEATURES = [
    "EPU_Level",
    "EPU_Change",
    "EPU_MA5",
    "EPU_MA20"
]

FEATURE_SETS = {
    "A_Baseline": MARKET_FEATURES,
    "B_GPR": MARKET_FEATURES + GPR_FEATURES,
    "C_EPU": MARKET_FEATURES + EPU_FEATURES,
    "D_Combined": (
        MARKET_FEATURES + GPR_FEATURES + EPU_FEATURES
    )
}

print("=" * 60)
print("FEATURE CONFIGURATIONS")
print("=" * 60)

for experiment, features in FEATURE_SETS.items():

    print(
        f"{experiment}: {len(features)} predictors"
    )

    print(features)
    print()

FEATURE CONFIGURATIONS
A_Baseline: 7 predictors
['Return_Lag1', 'Absolute_Return', 'RV_Historical_5D', 'RV_Historical_10D', 'RV_Historical_20D', 'RV_Historical_60D', 'RV_20D_Lag1']

B_GPR: 11 predictors
['Return_Lag1', 'Absolute_Return', 'RV_Historical_5D', 'RV_Historical_10D', 'RV_Historical_20D', 'RV_Historical_60D', 'RV_20D_Lag1', 'GPR_Level', 'GPR_Change', 'GPR_MA5', 'GPR_MA20']

C_EPU: 11 predictors
['Return_Lag1', 'Absolute_Return', 'RV_Historical_5D', 'RV_Historical_10D', 'RV_Historical_20D', 'RV_Historical_60D', 'RV_20D_Lag1', 'EPU_Level', 'EPU_Change', 'EPU_MA5', 'EPU_MA20']

D_Combined: 15 predictors
['Return_Lag1', 'Absolute_Return', 'RV_Historical_5D', 'RV_Historical_10D', 'RV_Historical_20D', 'RV_Historical_60D', 'RV_20D_Lag1', 'GPR_Level', 'GPR_Change', 'GPR_MA5', 'GPR_MA20', 'EPU_Level', 'EPU_Change', 'EPU_MA5', 'EPU_MA20']



In [7]:
# =========================================================
# STEP 5: CREATE MODELLING DATASET
# =========================================================

TARGET = "RV_Future_20D"

ALL_FEATURES = (
    MARKET_FEATURES +
    GPR_FEATURES +
    EPU_FEATURES
)

required_columns = (
    ["Date"] + ALL_FEATURES + [TARGET]
)

model_df = df[required_columns].copy()

print("Before removing missing values:", model_df.shape)

# Replace infinite values
model_df = model_df.replace(
    [np.inf, -np.inf],
    np.nan
)

# Remove incomplete observations
model_df = model_df.dropna(
    subset=ALL_FEATURES + [TARGET]
)

model_df = model_df.sort_values(
    "Date"
).reset_index(drop=True)

print("After removing missing values:", model_df.shape)

print("\nDate range:")
print(model_df["Date"].min())
print(model_df["Date"].max())

print("\nMissing values:")
print(model_df.isna().sum())

print("\nDuplicate dates:")
print(model_df["Date"].duplicated().sum())

# Validation
assert model_df["Date"].is_unique

assert model_df["Date"].is_monotonic_increasing

assert model_df[ALL_FEATURES + [TARGET]].notna().all().all()

assert (model_df[TARGET] >= 0).all()

print("\nALL FEATURE ENGINEERING CHECKS PASSED!")

# Save provisional dataset
output_file = (
    FINAL_DIR / "model_features_20d_provisional.csv"
)

model_df.to_csv(
    output_file,
    index=False
)

print("\nSaved to:")
print(output_file)

Before removing missing values: (4024, 17)
After removing missing values: (3944, 17)

Date range:
2010-03-31 00:00:00
2025-12-02 00:00:00

Missing values:
Date                 0
Return_Lag1          0
Absolute_Return      0
RV_Historical_5D     0
RV_Historical_10D    0
RV_Historical_20D    0
RV_Historical_60D    0
RV_20D_Lag1          0
GPR_Level            0
GPR_Change           0
GPR_MA5              0
GPR_MA20             0
EPU_Level            0
EPU_Change           0
EPU_MA5              0
EPU_MA20             0
RV_Future_20D        0
dtype: int64

Duplicate dates:
0

ALL FEATURE ENGINEERING CHECKS PASSED!

Saved to:
C:\Users\ibrah\RPDA8412 Research Project\data\final\model_features_20d_provisional.csv
